# AI Orchestrator — Colab Smoke Test (v12)

**이 노트북에서 검증하는 것**

1. **매 턴 동작** — `recent_messages` append + 기존 summary 읽어 프롬프트에 주입
2. **3턴마다 백그라운드 memory update** — user 3턴째마다 summary LLM이 병렬로 돌아 narrative/structured 재생성
3. **Colab에서도 fire-and-forget이 실제 실행** — `colab_loop_helper.py`의 영구 이벤트 루프 사용
4. **4개 라우팅 시나리오** — DIRECT_ANSWER, RETRIEVE_DOC, SEARCH_PREP_THEN_RETRIEVE, ASK_CLARIFICATION

**설계 요약**

```
매 턴 (사용자 critical path):
  user append → router → answer LLM → assistant append → save
         ↓ 응답 반환 (이 시점에 사용자는 답 받음)
  user_turn_count % 3 == 0 이면:
      create_task(memory_update) ← 뒷단에서 계속 실행
```

memory update는 사용자 응답 latency를 절대 막지 않음.

## 1. 레포 clone & 의존성 설치

In [ ]:
REPO_URL = "https://github.com/2026-Pretrained-Conversational-Model/ai-engine.git"
REPO_DIR = "/content/ai-orchestrator2"

import os
if not os.path.exists(REPO_DIR):
    !git clone -b refactor/pipeline {REPO_URL} {REPO_DIR}
else:
    print("repo already cloned")

In [ ]:
%pip install -q -U \
    "fastapi==0.115.0" "pydantic==2.9.2" "pydantic-settings==2.5.2" \
    "pypdf==5.0.0" "sentence-transformers==3.1.1" "faiss-cpu==1.8.0" \
    "langchain_text_splitters" \
    "transformers>=4.45" "accelerate>=0.34" "bitsandbytes>=0.46.1" \
    "boto3" "numpy<2"

In [ ]:
import torch
import bitsandbytes as bnb

print("torch:", torch.__version__)
print("bnb:", bnb.__version__)
print("cuda:", torch.cuda.is_available())
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

## 2. 환경변수 + sys.path (**반드시 app import 전에**)

In [ ]:
import os, sys

# local 백엔드 (HF 모델 in-process)
os.environ["LLM_BACKEND"] = "local"

# 한국어 임베딩 (Colab GPU 사용)
os.environ["EMBEDDING_MODEL_NAME"] = "jhgan/ko-sroberta-multitask"
os.environ["EMBEDDING_DIM"] = "768"
os.environ["EMBEDDING_DEVICE"] = "cuda"

# PDF 업로드 임시 저장소
os.environ["LOCAL_FILE_DIR"] = "/content/_orchestrator_files"

# 세션 메모리 캡 여유있게
os.environ["SESSION_MAX_BYTES"] = str(20 * 1024 * 1024)

# v12 핵심: memory update 정책
os.environ["MEMORY_UPDATE_EVERY_N_TURNS"] = "3"   # user 턴 3회마다 1번
os.environ["MEMORY_UPDATE_WINDOW_TURNS"] = "3"    # summary LLM에 직전 3턴 전체 입력
os.environ["ANSWER_MAX_NEW_TOKENS"]     = "200"   # latency + 언어 드리프트 제어

REPO_DIR = "/content/ai-orchestrator2"
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("env ready")

## 3. HuggingFace 모델 ID 설정

세 가지 role:
- **answer** — 최종 답변 LLM
- **router** — RAG 필요 여부 분류기
- **summary** (memory) — 3턴 누적 → narrative/structured JSON 생성

`None`으로 두면 해당 role은 휴리스틱/no-op 폴백.

In [ ]:
ANSWER_MODEL_ID  = "Qwen/Qwen2.5-7B-Instruct"
ROUTER_MODEL_ID  = "Qwen/Qwen2.5-3B-Instruct"
SUMMARY_MODEL_ID = "yeseul0-0/qwen2.5-3b-memory-summary-default_v0.3"

# T4 16GB에 7B + 3B + 3B 동시 로드를 위해 4bit 양자화
USE_4BIT = True

# summary 모델은 base Qwen 토크나이저를 써야 안정적
BASE_QWEN_TOKENIZER_ID = "Qwen/Qwen2.5-3B-Instruct"

## 4. 모델 로드

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig


def _load(model_id: str | None, tokenizer_id: str | None = None):
    if model_id is None:
        return None, None

    tok_id = tokenizer_id or model_id
    print(f"loading model={model_id}")
    print(f"loading tokenizer={tok_id}")

    tok = AutoTokenizer.from_pretrained(tok_id, trust_remote_code=True)
    tok.pad_token = tok.eos_token
    tok.padding_side = "right"

    kwargs = dict(device_map="auto", trust_remote_code=True)
    if USE_4BIT:
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
        )
    else:
        kwargs["torch_dtype"] = torch.float16

    model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
    model.config.pad_token_id = tok.pad_token_id
    model.config.eos_token_id = tok.eos_token_id
    if hasattr(model, "generation_config") and model.generation_config is not None:
        model.generation_config.pad_token_id = tok.pad_token_id
        model.generation_config.eos_token_id = tok.eos_token_id

    model.eval()
    print(f"  -> ready on {next(model.parameters()).device}")
    return model, tok


answer_model,  answer_tok  = _load(ANSWER_MODEL_ID)
router_model,  router_tok  = _load(ROUTER_MODEL_ID)
# summary 모델만 tokenizer를 base에서 가져온다 (학습 시 토크나이저 호환 이슈 회피)
summary_model, summary_tok = _load(SUMMARY_MODEL_ID, tokenizer_id=BASE_QWEN_TOKENIZER_ID)

## 5. LocalModelRegistry 등록 (1회)

`chat()` 호출마다 재등록하지 않고 **노트북 시작 시 1회만**. 이후 pipeline은 등록된 모델을 계속 재사용.

In [ ]:
from app.services.llm.local_registry import LocalModelRegistry

LocalModelRegistry.clear()  # 노트북 셀 재실행 안전

if answer_model is not None:
    LocalModelRegistry.register(
        "answer", answer_model, answer_tok,
        device="cuda", max_new_tokens=200,
    )

if router_model is not None:
    LocalModelRegistry.register(
        "router", router_model, router_tok,
        device="cuda", max_new_tokens=60,
    )

# summary 모델은 "memory" role로 등록
# (memory_state_generator가 memory → summary 순서로 role을 찾음)
if summary_model is not None:
    LocalModelRegistry.register(
        "memory", summary_model, summary_tok,
        device="cuda", max_new_tokens=400,
    )

print("registered roles:", LocalModelRegistry.list_roles())

## 6. 영구 이벤트 루프 헬퍼 로드 (핵심)

기존 `asyncio.run_until_complete()` 방식은 `pipeline.run()`이 끝나면 루프도 종료 → `finalize`가 `create_task()`로 스케줄한 **memory update task가 고아**가 됨.

`colab_loop_helper.py`는 daemon 스레드에 영구 이벤트 루프를 띄워서 `pipeline.run()`이 끝나도 백그라운드 task가 계속 실행되게 함.

In [ ]:
# 영구 루프 헬퍼
from notebooks.colab_loop_helper import chat, dump_session, wait_for_background, run_coro

# 임베딩 모델 warmup (원래 FastAPI lifespan에서 하던 일)
from app.services.embedding.embedding_singleton import EmbeddingSingleton
EmbeddingSingleton.warmup()

# 설정 확인
from app.core.config import settings
print("LLM_BACKEND:", settings.LLM_BACKEND)
print("MEMORY_UPDATE_EVERY_N_TURNS:", settings.MEMORY_UPDATE_EVERY_N_TURNS)
print("MEMORY_UPDATE_WINDOW_TURNS:", settings.MEMORY_UPDATE_WINDOW_TURNS)
print("ANSWER_MAX_NEW_TOKENS:", settings.ANSWER_MAX_NEW_TOKENS)
print("registered roles:", LocalModelRegistry.list_roles())

## 시나리오 A — DIRECT_ANSWER (PDF 없음)

user 턴 3회까지 진행해서 **3턴째에서 memory update task가 스케줄되는지** 확인.

예상 로그 흐름:
- 턴 1: `[MEMORY_POLICY] should_update=False reason=skip(user_turn=1, every=3)`
- 턴 2: `[MEMORY_POLICY] should_update=False reason=skip(user_turn=2, every=3)`
- 턴 3: `[MEMORY_POLICY] should_update=True reason=every_3_turns(user_turn=3)` → `scheduled background task`
- 잠시 후: `[MEMORY_UPDATE] start ... done` (background에서 완료)

In [ ]:
sid_a = "test-a-1"

r1 = chat(sid_a, "안녕, 내 이름은 김예슬이야.")
print("=== turn 1 ===")
print("answer_type:", r1["answer_type"])
print("answer     :", r1["answer"])
print()

r2 = chat(sid_a, "내 이름이 뭐라고?")
print("=== turn 2 ===")
print("answer_type:", r2["answer_type"])
print("answer     :", r2["answer"])
print()

r3 = chat(sid_a, "RAG가 뭐야?")
print("=== turn 3 (memory update 스케줄되는 턴) ===")
print("answer_type:", r3["answer_type"])
print("answer     :", r3["answer"])

In [ ]:
# 턴 3의 memory update background task가 끝날 때까지 기다림 (최대 60초)
wait_for_background(timeout=60.0)

# 세션 덤프 — summary.narrative / structured가 채워졌는지 확인
s = dump_session(sid_a)
print("narrative:", s["conversation"]["summary"]["narrative"])
print("structured:", s["conversation"]["summary"]["structured"])

In [ ]:
# 턴 4, 5, 6 — 두번째 3턴 사이클
r4 = chat(sid_a, "챗봇 아키텍처를 만들고 싶은데, 어느 부분에 RAG를 쓰면 좋을까?")
print("=== turn 4 ===")
print("answer     :", r4["answer"])
print()

r5 = chat(sid_a, "좀더 쉽게 설명해줘.")
print("=== turn 5 ===")
print("answer     :", r5["answer"])
print()

r6 = chat(sid_a, "내 이름이 뭐라고 했더라?")
print("=== turn 6 (memory update 2번째 스케줄) ===")
print("answer     :", r6["answer"])

wait_for_background(timeout=60.0)
print("\n--- after 2nd memory update ---")
s = dump_session(sid_a)
print("narrative:", s["conversation"]["summary"]["narrative"])
print("structured:", s["conversation"]["summary"]["structured"])

## 시나리오 B — RETRIEVE_DOC (PDF 업로드)

PDF 업로드 + 문서 관련 질문 → 라우터가 `RETRIEVE_DOC` 반환 → FAISS retrieve → RAG 답변.

**PDF 첨부 시 중국어 답변 나오는 문제**는 `pdf_summarizer.py`가 원문 140자를 그대로 프롬프트에 박던 stub이 원인이었는데, v11부터 한국어 한 문장 요약(또는 메타정보 폴백)으로 바뀌었어. PDF가 영문/중문이어도 `doc_summary.one_line`은 항상 한국어.

In [ ]:
from google.colab import files
uploaded = files.upload()   # 파일 선택 창
pdf_name = list(uploaded.keys())[0]
pdf_path = f"/content/{pdf_name}"
with open(pdf_path, "wb") as f:
    f.write(uploaded[pdf_name])
print("saved to", pdf_path)

In [ ]:
sid_b = "test-b-1"

r1 = chat(sid_b, "이 문서가 뭐에 대한 거야? 간단히 요약해줘.",
          file_path=pdf_path, file_name=pdf_name)
print("=== turn 1 (PDF 첨부) ===")
print("answer_type:", r1["answer_type"])
print("answer     :", r1["answer"])
print()

r2 = chat(sid_b, "이 문서에서 핵심만 3개 뽑아줘.")
print("=== turn 2 ===")
print("answer_type:", r2["answer_type"])
print("answer     :", r2["answer"])
print()

r3 = chat(sid_b, "거기서 가장 중요한 건 뭐야?")
print("=== turn 3 (memory update 스케줄) ===")
print("answer_type:", r3["answer_type"])
print("answer     :", r3["answer"])

wait_for_background(timeout=60.0)
print("\n--- after memory update ---")
s = dump_session(sid_b)
print("narrative:", s["conversation"]["summary"]["narrative"])

## 시나리오 C — SEARCH_PREP_THEN_RETRIEVE (참조 표현 + PDF)

PDF가 세션에 붙어있는 상태에서 `"그거"`, `"아까 그 표"` 같은 지시어로 질문 → 라우터가 `SEARCH_PREP_THEN_RETRIEVE` 반환 → 쿼리 재작성 후 retrieve.

In [ ]:
# 시나리오 B에 이어서 같은 세션에 참조 표현 질문
r4 = chat(sid_b, "그거 표로 정리해줘.")
print("=== turn 4 (참조 표현) ===")
print("answer_type:", r4["answer_type"])
print("answer     :", r4["answer"])
print()

# 세션에 어떤 router 결정이 찍혔는지 확인
s = dump_session(sid_b)
print("last_router_decision:", s["runtime_state"]["last_router_decision"])
print("last_answer_type    :", s["runtime_state"]["last_answer_type"])

## 시나리오 D — ASK_CLARIFICATION (문맥 부족)

빈 세션 + PDF 없음 + 지시어만 있는 짧은 질문 → 라우터가 `ASK_CLARIFICATION` 반환 → **LLM 호출 자체를 스킵**하고 고정 재질문 메시지만 반환.

In [ ]:
sid_d = "test-d-1"

r = chat(sid_d, "그거 뭐야?")
print("answer_type:", r["answer_type"])
print("answer     :", r["answer"])
# LLM 호출 없이 즉시 반환되어야 함

## 메모리 업데이트 동작 검증

아래 로그에서 `[MEMORY_POLICY]`과 `[MEMORY_UPDATE]`를 확인하세요.

- `skip(user_turn=1, every=3)` — 턴 1, 2는 스케줄 안 됨 ✓
- `scheduled background task` — 턴 3에서 스케줄 ✓
- `[MEMORY_UPDATE] start` → `done` — background에서 실제 실행 ✓
- pipeline.run()이 응답 반환한 뒤에 [MEMORY_UPDATE] done 로그가 찍히면 성공 ✓

만약 [MEMORY_UPDATE] 로그가 안 찍히면:
- `colab_loop_helper`가 import 안 됐거나
- 3턴이 아직 안 쌓였거나
- `summary`/`memory` role이 등록 안 됨

In [ ]:
# 현재 registered 상태 확인
print("roles:", LocalModelRegistry.list_roles())
print("has memory:", LocalModelRegistry.has("memory"))
print("has summary:", LocalModelRegistry.has("summary"))

# 전체 세션 덤프
import json
print("\n--- session A ---")
print(json.dumps(dump_session(sid_a), ensure_ascii=False, indent=2)[:2000])

## 세션 정리

In [ ]:
from app.services.session.session_cleaner import purge_session

for sid in [sid_a, sid_b, sid_d]:
    run_coro(purge_session(sid, reason="test_done"))
print("purged")